# 01 - Data Collection & Ingestion


In [1]:
import os
import glob
import xml.etree.ElementTree as ET

from pyspark.sql import SparkSession
from pyspark.sql.functions import *


In [2]:
import os
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

In [3]:
import sys
from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Bus_Service_Risk_Classification")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.local.ip", "127.0.0.1")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")


In [4]:
print("Default Parallelism:", spark.sparkContext.defaultParallelism)

Default Parallelism: 16


In [5]:
print(spark.sparkContext.uiWebUrl)

http://127.0.0.1:4043


In [6]:
project_path = r"D:\BigDataCoursework"

raw_path = os.path.join(project_path, "Data", "raw")

timetable_path   = os.path.join(raw_path, "timetable")
vehicle_path     = os.path.join(raw_path, "vehicle_location")
disruption_path  = os.path.join(raw_path, "disruption")

print(timetable_path)
print(vehicle_path)
print(disruption_path)


D:\BigDataCoursework\Data\raw\timetable
D:\BigDataCoursework\Data\raw\vehicle_location
D:\BigDataCoursework\Data\raw\disruption


## 1. Timetable catalogue (TransXChange XML)

In [7]:
xml_pattern = os.path.join(timetable_path, "**", "*.xml")
xml_files = glob.glob(xml_pattern, recursive=True)
print("Total Timetable XML Files:", len(xml_files))


Total Timetable XML Files: 423


In [8]:
NS = {"tx": "http://www.transxchange.org.uk/"}


In [9]:
journeys = []

for file in xml_files:
    try:
        tree = ET.parse(file)
        root = tree.getroot()

        for journey in root.findall(".//tx:VehicleJourney", NS):
            journeys.append((
                journey.findtext("tx:VehicleJourneyCode", default="Unknown", namespaces=NS),
                journey.findtext("tx:OperatorRef", default="Unknown", namespaces=NS),
                journey.findtext("tx:ServiceRef", default="Unknown", namespaces=NS),
                journey.findtext("tx:LineRef", default="Unknown", namespaces=NS),
                journey.findtext("tx:JourneyPatternRef", default="Unknown", namespaces=NS),
                journey.findtext("tx:DepartureTime", default="00:00:00", namespaces=NS),
            ))
    except Exception:
        continue

print("Total Vehicle Journeys:", len(journeys))


Total Vehicle Journeys: 48481


In [10]:
columns = ["Journey_ID", "Operator_ID", "Service_Code", "Line_Name",
           "JourneyPatternRef", "Departure_Time"]

journeys_df = spark.createDataFrame(journeys, columns)
journeys_df.show(5, truncate=False)


+----------+-----------+------------+-------------------+-----------------+--------------+
|Journey_ID|Operator_ID|Service_Code|Line_Name          |JourneyPatternRef|Departure_Time|
+----------+-----------+------------+-------------------+-----------------+--------------+
|VJ_1      |AMGR       |PD0001892:94|AMGR:PD0001892:94:6|JP_1             |09:40:00      |
|VJ_2      |AMGR       |PD0001892:94|AMGR:PD0001892:94:6|JP_2             |10:55:00      |
|VJ_3      |AMGR       |PD0001892:94|AMGR:PD0001892:94:6|JP_2             |13:10:00      |
|VJ_4      |AMGR       |PD0001892:94|AMGR:PD0001892:94:6|JP_3             |12:00:00      |
|VJ_5      |AMGR       |PD0001892:94|AMGR:PD0001892:94:6|JP_3             |13:45:00      |
+----------+-----------+------------+-------------------+-----------------+--------------+
only showing top 5 rows



In [11]:
timing_links = []

for file in xml_files:
    try:
        tree = ET.parse(file)
        root = tree.getroot()

        operator = root.findtext(".//tx:Operator/tx:NationalOperatorCode", default="Unknown", namespaces=NS)

        journey_patterns = {}
        for jp in root.findall(".//tx:JourneyPattern", NS):
            jp_id = jp.attrib.get("id")
            journey_patterns[jp_id] = jp.findtext("tx:JourneyPatternSectionRefs", default="", namespaces=NS)

        sections = {}
        for section in root.findall(".//tx:JourneyPatternSection", NS):
            section_id = section.attrib.get("id")
            sections[section_id] = section.findall("tx:JourneyPatternTimingLink", NS)

        for journey in root.findall(".//tx:VehicleJourney", NS):
            journey_id = journey.findtext("tx:VehicleJourneyCode", default="Unknown", namespaces=NS)
            pattern = journey.findtext("tx:JourneyPatternRef", default="", namespaces=NS)
            section_id = journey_patterns.get(pattern)

            if section_id in sections:
                for link in sections[section_id]:
                    timing_links.append((
                        journey_id,
                        operator,
                        link.findtext("tx:From/tx:StopPointRef", default="Unknown", namespaces=NS),
                        link.findtext("tx:To/tx:StopPointRef", default="Unknown", namespaces=NS),
                        link.findtext("tx:RunTime", default="PT0S", namespaces=NS),
                    ))
    except Exception:
        continue

print("Total Timing Link Records:", len(timing_links))


Total Timing Link Records: 1825566


In [12]:
timing_df = spark.createDataFrame(
    timing_links,
    ["Journey_ID", "Operator_ID", "From_Stop", "To_Stop", "Run_Time"]
)
print("Timing Link Records:", timing_df.count())
timing_df.show(5, truncate=False)


Timing Link Records: 1825566
+----------+-----------+-----------+-----------+--------+
|Journey_ID|Operator_ID|From_Stop  |To_Stop    |Run_Time|
+----------+-----------+-----------+-----------+--------+
|VJ_1      |AMGR       |4200F066400|4200F066401|PT2M    |
|VJ_1      |AMGR       |4200F066401|4200F066000|PT2M    |
|VJ_1      |AMGR       |4200F066000|4200F147820|PT2M    |
|VJ_1      |AMGR       |4200F147820|4200F067203|PT1M    |
|VJ_1      |AMGR       |4200F067203|4200F065501|PT2M    |
+----------+-----------+-----------+-----------+--------+
only showing top 5 rows



In [13]:
route_complexity = []

for file in xml_files:
    try:
        tree = ET.parse(file)
        root = tree.getroot()

        complexity = len(root.findall(".//tx:JourneyPatternTimingLink", NS))

        for journey in root.findall(".//tx:VehicleJourney", NS):
            route_complexity.append((
                journey.findtext("tx:VehicleJourneyCode", default="Unknown", namespaces=NS),
                journey.findtext("tx:OperatorRef", default="Unknown", namespaces=NS),
                complexity,
            ))
    except Exception:
        continue

complexity_df = spark.createDataFrame(
    route_complexity, ["Journey_ID", "Operator_ID", "Route_Complexity"]
)

complexity_df_clean = (
    complexity_df.groupBy("Journey_ID", "Operator_ID")
    .agg(max("Route_Complexity").alias("Route_Complexity"))
)
print("Complexity Records:", complexity_df_clean.count())


Complexity Records: 35214


## 2. Disruptions catalogue (SIRI-SX XML)




In [14]:
SIRI_NS = {"siri": "http://www.siri.org.uk/siri"}

disruption_files = glob.glob(os.path.join(disruption_path, "**", "*.xml"), recursive=True)
print("Total Disruption XML Files:", len(disruption_files))


Total Disruption XML Files: 1


In [15]:
def inspect_one_disruption_file():
    """Run this once to check the real tag names in your disruption files
    before trusting the parser below."""
    if not disruption_files:
        print("No disruption files found - check disruption_path.")
        return
    tree = ET.parse(disruption_files[0])
    for elem in tree.getroot().iter():
        tag = elem.tag.split('}')[-1]
        print(tag, '->', (elem.text or '').strip()[:40])



In [16]:
disruptions = []

for file in disruption_files:
    try:
        tree = ET.parse(file)
        root = tree.getroot()

        for situation in root.findall(".//siri:PtSituationElement", SIRI_NS):

            situation_id = situation.findtext("siri:SituationNumber", default="Unknown", namespaces=SIRI_NS)
            severity     = situation.findtext("siri:Severity", default="unknown", namespaces=SIRI_NS)
            reason       = situation.findtext("siri:ReasonName", default="Unknown", namespaces=SIRI_NS)
            summary      = situation.findtext("siri:Summary", default="", namespaces=SIRI_NS)
            start_time   = situation.findtext(".//siri:ValidityPeriod/siri:StartTime", default=None, namespaces=SIRI_NS)
            end_time     = situation.findtext(".//siri:ValidityPeriod/siri:EndTime", default=None, namespaces=SIRI_NS)

            affected_lines = situation.findall(".//siri:AffectedLine", SIRI_NS)

            if not affected_lines:
                disruptions.append((situation_id, "Unknown", "Unknown", severity, reason, summary, start_time, end_time))
            else:
                for line in affected_lines:
                    operator_ref = line.findtext(".//siri:OperatorRef", default="Unknown", namespaces=SIRI_NS)
                    line_ref     = line.findtext(".//siri:LineRef", default="Unknown", namespaces=SIRI_NS)
                    disruptions.append((situation_id, operator_ref, line_ref, severity, reason, summary, start_time, end_time))

    except Exception:
        continue

print("Total Disruption Records:", len(disruptions))


Total Disruption Records: 1530


In [17]:
disruption_columns = ["Situation_ID", "Operator_ID", "Line_Ref", "Severity",
                       "Reason", "Summary", "Start_Time", "End_Time"]

disruption_df = spark.createDataFrame(disruptions, disruption_columns) if disruptions else \
    spark.createDataFrame([], schema=", ".join(f"{c} string" for c in disruption_columns))

disruption_df.show(10, truncate=False)
print("Disruption records:", disruption_df.count())


+------------------------------------+-----------+---------+--------+-------+--------------------------------------------------------------------------+------------------------+------------------------+
|Situation_ID                        |Operator_ID|Line_Ref |Severity|Reason |Summary                                                                   |Start_Time              |End_Time                |
+------------------------------------+-----------+---------+--------+-------+--------------------------------------------------------------------------+------------------------+------------------------+
|18224249-29cf-4f65-a023-11067e7c2b6f|Unknown    |Unknown  |unknown |Unknown|Live Traffic Update: York Road One Way Closure                            |2024-09-01T08:00:00.000Z|NULL                    |
|2b9e1d8f-b0ee-43a7-8ca5-7334d6fc4587|YSQU       |9        |unknown |Unknown|Horsforth Vale, Bletchley Avenue, Bletchley Road and Low Hall Road (Leeds)|2024-11-04T08:30:00.000Z|NULL       

## 3. Feature engineering on the timetable data 

In [18]:
journeys_df = journeys_df.join(complexity_df_clean, on=["Journey_ID", "Operator_ID"], how="left")

timing_features = timing_df.groupBy("Journey_ID", "Operator_ID").agg(
    countDistinct("To_Stop").alias("Number_of_Stops"),
    first("From_Stop").alias("First_Stop"),
    first("To_Stop").alias("First_To_Stop"),
)

journeys_df = journeys_df.join(timing_features, on=["Journey_ID", "Operator_ID"], how="left")

operator_stats = journeys_df.groupBy("Operator_ID").agg(
    count("*").alias("Total_Journeys"),
    avg("Route_Complexity").alias("Average_Route_Complexity"),
)

journeys_df = journeys_df.join(operator_stats, on="Operator_ID", how="left")

journeys_df = journeys_df.withColumn(
    "Departure_Hour", substring("Departure_Time", 1, 2).cast("integer")
).withColumn(
    "Peak_Hour",
    when((col("Departure_Hour") >= 7) & (col("Departure_Hour") <= 9), 1)
    .when((col("Departure_Hour") >= 16) & (col("Departure_Hour") <= 18), 1)
    .otherwise(0)
)

journeys_df = journeys_df.withColumn(
    "Risk_Score",
    (col("Route_Complexity") * 0.002) + (col("Number_of_Stops") * 0.60)
    + (col("Peak_Hour") * 12) + ((24 - col("Departure_Hour")) * 0.25)
).withColumn(
    "Service_Risk",
    when(col("Risk_Score") >= 70, 2).when(col("Risk_Score") >= 35, 1).otherwise(0)
)

journeys_df.groupBy("Service_Risk").count().show()


+------------+-----+
|Service_Risk|count|
+------------+-----+
|           1| 2190|
|           2|  274|
|           0|46017|
+------------+-----+



## 4. Join the timetable data with the Disruptions catalogue




In [19]:
disruption_flag = (
    disruption_df
    .groupBy("Operator_ID")
    .agg(count("*").alias("Disruption_Count"))
    .withColumn("Has_Disruption", lit(1))
)

journeys_df = journeys_df.join(disruption_flag, on="Operator_ID", how="left")
journeys_df = journeys_df.fillna({"Disruption_Count": 0, "Has_Disruption": 0})

print("Journeys with a disruption flag joined in:")
journeys_df.groupBy("Has_Disruption").count().show()


Journeys with a disruption flag joined in:
+--------------+-----+
|Has_Disruption|count|
+--------------+-----+
|             1| 1808|
|             0|46673|
+--------------+-----+



In [20]:
final_df = timing_df.join(
    journeys_df.select(
        "Journey_ID", "Operator_ID", "Service_Code", "Line_Name", "JourneyPatternRef",
        "Departure_Time", "Route_Complexity", "Number_of_Stops", "Total_Journeys",
        "Average_Route_Complexity", "Risk_Score", "Departure_Hour", "Peak_Hour",
        "Service_Risk", "Disruption_Count", "Has_Disruption",
    ),
    on=["Journey_ID", "Operator_ID"],
    how="inner",
)

print("Final Big Data Dataset:", final_df.count())
final_df.printSchema()


Final Big Data Dataset: 227010
root
 |-- Journey_ID: string (nullable = true)
 |-- Operator_ID: string (nullable = true)
 |-- From_Stop: string (nullable = true)
 |-- To_Stop: string (nullable = true)
 |-- Run_Time: string (nullable = true)
 |-- Service_Code: string (nullable = true)
 |-- Line_Name: string (nullable = true)
 |-- JourneyPatternRef: string (nullable = true)
 |-- Departure_Time: string (nullable = true)
 |-- Route_Complexity: long (nullable = true)
 |-- Number_of_Stops: long (nullable = true)
 |-- Total_Journeys: long (nullable = true)
 |-- Average_Route_Complexity: double (nullable = true)
 |-- Risk_Score: double (nullable = true)
 |-- Departure_Hour: integer (nullable = true)
 |-- Peak_Hour: integer (nullable = false)
 |-- Service_Risk: integer (nullable = false)
 |-- Disruption_Count: long (nullable = false)
 |-- Has_Disruption: integer (nullable = false)



## 5. Partitioning, caching & parallelism (evidence for the report)




In [21]:
print("Partitions BEFORE repartition:", final_df.rdd.getNumPartitions())

final_df = final_df.repartition(spark.sparkContext.defaultParallelism)
final_df.cache()
final_df.count()  # materialise the cache

print("Partitions AFTER repartition:", final_df.rdd.getNumPartitions())
print("Cores used (defaultParallelism):", spark.sparkContext.defaultParallelism)


Partitions BEFORE repartition: 17
Partitions AFTER repartition: 16
Cores used (defaultParallelism): 16


In [22]:
final_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in final_df.columns
]).show()


+----------+-----------+---------+-------+--------+------------+---------+-----------------+--------------+----------------+---------------+--------------+------------------------+----------+--------------+---------+------------+----------------+--------------+
|Journey_ID|Operator_ID|From_Stop|To_Stop|Run_Time|Service_Code|Line_Name|JourneyPatternRef|Departure_Time|Route_Complexity|Number_of_Stops|Total_Journeys|Average_Route_Complexity|Risk_Score|Departure_Hour|Peak_Hour|Service_Risk|Disruption_Count|Has_Disruption|
+----------+-----------+---------+-------+--------+------------+---------+-----------------+--------------+----------------+---------------+--------------+------------------------+----------+--------------+---------+------------+----------------+--------------+
|         0|          0|        0|      0|       0|           0|        0|                0|             0|               0|              0|             0|                       0|         0|             0|        

In [23]:
processed_path = os.path.join(project_path, "Data", "processed")

final_df.write.mode("overwrite").parquet(os.path.join(processed_path, "bus_risk_dataset.parquet"))
operator_stats.write.mode("overwrite").parquet(os.path.join(processed_path, "operator_stats.parquet"))
disruption_df.write.mode("overwrite").parquet(os.path.join(processed_path, "disruptions.parquet"))
timing_df.write.mode("overwrite").parquet(os.path.join(processed_path, "timing_links.parquet"))

print("Saved: bus_risk_dataset.parquet, operator_stats.parquet, disruptions.parquet, timing_links.parquet")
print("Final row count:", final_df.count(), "| Columns:", len(final_df.columns))


Saved: bus_risk_dataset.parquet, operator_stats.parquet, disruptions.parquet, timing_links.parquet
Final row count: 227010 | Columns: 19


In [24]:
final_df.unpersist()
spark.stop()
print("Spark session stopped successfully.")


Spark session stopped successfully.
